# Evaluasi Argumen Bertingkat — MASSIVE-Agents

Analisis detail argumen pada 2.937 prediksi baseline development. Skor BFCL AST tetap menjadi hasil evaluasi utama. Analisis bertingkat digunakan untuk menjelaskan letak perbedaan argumen terhadap ground truth.

## 1. Konfigurasi

Analisis menggunakan hasil `ast_per_prediction.csv` dari evaluasi sebelumnya. Penilaian dilakukan pada prediksi yang memilih fungsi dengan benar. Ground truth, prediksi, dan skor AST asli tidak diubah.

In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path

SEED = 20261110
SAMPEL_AUDIT = 60
EXPECTED = 2937


def cari_project():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"D:\KP-Agentic-AI")]
    for root in candidates:
        if (root / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv").exists():
            return root
        if (root / "ast_per_prediction.csv").exists():
            return root
    raise FileNotFoundError("ast_per_prediction.csv tidak ditemukan di lokasi project.")


ROOT = cari_project()
INPUT = ROOT / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv"
if not INPUT.exists():
    INPUT = ROOT / "ast_per_prediction.csv"
OUTPUT = ROOT / "results" / "evaluasi_argumen_bertingkat"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Input:", INPUT)
print("Output:", OUTPUT)

Project: D:\KP-Agentic-AI
Input: D:\KP-Agentic-AI\results\bfcl_ast_evaluation\ast_per_prediction.csv
Output: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat


## 2. Data hasil evaluasi AST

Setiap baris memuat nama fungsi, argumen prediksi, ground truth, skor AST, dan skor exact match internal. Pasangan model–record diperiksa agar tidak ada prediksi ganda.

In [2]:
def baca_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def boolean(value):
    return value is True or str(value).strip().lower() == "true"


rows = baca_csv(INPUT)
required = {"model", "row_key", "language_code", "group_id", "question",
            "target_function", "prediction_function", "prediction_arguments_json",
            "ground_truth_json", "full_match_internal", "bfcl_ast_correct"}
missing_columns = required - set(rows[0])
if missing_columns:
    raise ValueError(f"Kolom tidak tersedia: {sorted(missing_columns)}")
if len(rows) != EXPECTED:
    raise ValueError(f"Jumlah prediksi berbeda: {len(rows)} (seharusnya {EXPECTED})")
if len({(r["model"], r["row_key"]) for r in rows}) != len(rows):
    raise ValueError("Ada duplikasi pasangan model dan record")

MODELS = sorted({r["model"] for r in rows})
assert len(MODELS) == 3
for model in MODELS:
    assert sum(r["model"] == model for r in rows) == 979

print("Prediksi:", len(rows))
print("Model:", len(MODELS))
print("AST benar:", sum(boolean(r["bfcl_ast_correct"]) for r in rows))
print("Exact match internal:", sum(boolean(r["full_match_internal"]) for r in rows))

Prediksi: 2937
Model: 3
AST benar: 716
Exact match internal: 701


## 3. Aturan penilaian bertingkat

Pemeriksaan dibagi menjadi format JSON, pemilihan fungsi, struktur parameter, serta nilai argumen. Parameter dengan GT yang memuat nilai kosong (`""`) diperlakukan sebagai parameter yang boleh tidak diisi, mengikuti aturan exact match internal sebelumnya.

Kecocokan nilai dipisahkan menjadi **teks persis** dan **normalisasi sederhana** (huruf besar/kecil serta spasi). Perbedaan tanda baca atau parafrase hanya menjadi kandidat audit, bukan jawaban yang otomatis benar. Status AST resmi tidak dihitung ulang.

In [3]:
def normalisasi_internal(value):
    return " ".join(value.split()).casefold()


def normalisasi_format(value):
    # Hanya untuk MENANDAI kandidat perbedaan penulisan, bukan menambah skor benar.
    v = unicodedata.normalize("NFKC", value)
    v = v.casefold().strip()
    v = re.sub(r"[\s_\-]+", " ", v)
    v = re.sub(r"[.,!?;:]+$", "", v).strip()
    return v


def kecocokan_nilai(value, allowed):
    if value in allowed:
        return "teks_persis", ""
    if normalisasi_internal(value) in {normalisasi_internal(s) for s in allowed}:
        return "normalisasi_internal", ""
    if normalisasi_format(value) in {normalisasi_format(s) for s in allowed}:
        return "beda_format_kandidat", "perlu_review"
    alternatives = [s for s in allowed if isinstance(s, str) and len(s.strip()) >= 4]
    similarity = max(
        (SequenceMatcher(None, normalisasi_internal(value), normalisasi_internal(s)).ratio()
         for s in alternatives),
        default=0,
    )
    if len(value.strip()) >= 4 and similarity >= 0.84:
        return "mirip_teks_kandidat", "perlu_review"
    return "nilai_berbeda", ""


def analisis_prediksi(row):
    gt = json.loads(row["ground_truth_json"])
    gt_fn, allowed_args = next(iter(gt.items()))
    predicted_fn = row["prediction_function"]
    ast_ok = boolean(row["bfcl_ast_correct"])
    internal_ok = boolean(row["full_match_internal"])
    detail = []

    try:
        pred = json.loads(row["prediction_arguments_json"])
    except (json.JSONDecodeError, TypeError):
        pred = None
    valid = isinstance(pred, dict) and bool(predicted_fn)
    function_ok = valid and predicted_fn == gt_fn

    item = {
        "model": row["model"], "row_key": row["row_key"],
        "language_code": row["language_code"], "group_id": row["group_id"],
        "question": row["question"], "target_function": gt_fn,
        "prediction_function": predicted_fn,
        "prediction_arguments_json": row["prediction_arguments_json"],
        "ground_truth_json": row["ground_truth_json"],
        "format_valid": valid, "function_correct": function_ok,
        "bfcl_ast_correct": ast_ok, "full_match_internal": internal_ok,
        "ast_error_type": row.get("ast_error_type", ""),
        "ast_error_message": row.get("ast_error_message", ""),
        "jumlah_gt_required": sum("" not in values for values in allowed_args.values()),
        "jumlah_parameter_prediksi": len(pred) if valid else 0,
        "parameter_tidak_dikenal": 0, "parameter_wajib_hilang": 0,
        "parameter_opsional_hilang": 0, "tipe_nilai_salah": 0,
        "nilai_berbeda": 0, "kandidat_format": 0, "kandidat_mirip_teks": 0,
        "nilai_teks_persis": 0, "nilai_normalisasi_internal": 0,
        "kunci_valid": False, "semua_nilai_persis": False,
        "semua_nilai_normalisasi": False,
        "kategori_bertingkat": "",
    }

    if not valid:
        item["kategori_bertingkat"] = "format_tidak_valid"
        return item, detail
    if not function_ok:
        item["kategori_bertingkat"] = "fungsi_salah"
        return item, detail

    unexpected = set(pred) - set(allowed_args)
    required_missing = {k for k, alternatives in allowed_args.items()
                        if "" not in alternatives and k not in pred}
    optional_missing = {k for k, alternatives in allowed_args.items()
                        if "" in alternatives and k not in pred}
    item["parameter_tidak_dikenal"] = len(unexpected)
    item["parameter_wajib_hilang"] = len(required_missing)
    item["parameter_opsional_hilang"] = len(optional_missing)

    for key, allowed in allowed_args.items():
        value = pred.get(key)
        if key not in pred:
            jenis = "opsional_tidak_diisi" if "" in allowed else "wajib_tidak_diisi"
            review_flag = ""
        elif not isinstance(value, str):
            jenis = "tipe_nilai_salah"
            review_flag = ""
            item["tipe_nilai_salah"] += 1
        else:
            jenis, review_flag = kecocokan_nilai(value, allowed)
            if jenis == "teks_persis":
                item["nilai_teks_persis"] += 1
            elif jenis == "normalisasi_internal":
                item["nilai_normalisasi_internal"] += 1
            elif jenis == "beda_format_kandidat":
                item["kandidat_format"] += 1
            elif jenis == "mirip_teks_kandidat":
                item["kandidat_mirip_teks"] += 1
            else:
                item["nilai_berbeda"] += 1
        detail.append({
            "model": row["model"], "row_key": row["row_key"],
            "language_code": row["language_code"], "group_id": row["group_id"],
            "target_function": gt_fn, "parameter": key,
            "nilai_prediksi": json.dumps(value, ensure_ascii=False) if key in pred else "<tidak_ada>",
            "nilai_gt_diterima": json.dumps(allowed, ensure_ascii=False),
            "kategori": jenis, "perlu_review": review_flag,
        })

    for key in sorted(unexpected):
        detail.append({
            "model": row["model"], "row_key": row["row_key"],
            "language_code": row["language_code"], "group_id": row["group_id"],
            "target_function": gt_fn, "parameter": key,
            "nilai_prediksi": json.dumps(pred[key], ensure_ascii=False),
            "nilai_gt_diterima": "<tidak_ada_dalam_GT>",
            "kategori": "parameter_tidak_dikenal", "perlu_review": "",
        })

    type_error = item["tipe_nilai_salah"] > 0
    keys_ok = not unexpected and not required_missing
    item["kunci_valid"] = keys_ok
    values_strict = keys_ok and not type_error and all(
        not isinstance(pred.get(k), str) or pred[k] in allowed_args[k]
        for k in pred if k in allowed_args
    )
    values_norm = keys_ok and not type_error and all(
        isinstance(pred[k], str) and
        normalisasi_internal(pred[k]) in {normalisasi_internal(x) for x in allowed_args[k]}
        for k in pred if k in allowed_args
    )
    item["semua_nilai_persis"] = values_strict
    item["semua_nilai_normalisasi"] = values_norm

    if unexpected or required_missing or type_error:
        item["kategori_bertingkat"] = "struktur_parameter_bermasalah"
    elif values_strict:
        item["kategori_bertingkat"] = "cocok_teks_persis"
    elif values_norm:
        item["kategori_bertingkat"] = "cocok_normalisasi_internal"
    elif item["kandidat_format"] or item["kandidat_mirip_teks"]:
        item["kategori_bertingkat"] = "kandidat_perbedaan_penulisan"
    else:
        item["kategori_bertingkat"] = "nilai_argumen_berbeda"

    return item, detail


# Uji sederhana untuk aturan normalisasi
assert kecocokan_nilai("Jakarta", ["Jakarta"])[0] == "teks_persis"
assert kecocokan_nilai(" JAKARTA ", ["Jakarta"])[0] == "normalisasi_internal"
assert kecocokan_nilai("jakarta.", ["Jakarta"])[0] == "beda_format_kandidat"
assert kecocokan_nilai("Bandung", ["Jakarta"])[0] == "nilai_berbeda"
print("Uji aturan penilaian: PASS")

Uji aturan penilaian: PASS


## 4. Hasil per prediksi dan parameter

Hasil prediksi disimpan bersama kategori kesalahannya. Tabel parameter mencatat nilai GT yang diterima, nilai prediksi, dan kemungkinan perbedaan penulisan.

In [4]:
hasil_prediksi = []
hasil_parameter = []
for row in rows:
    result, detail = analisis_prediksi(row)
    hasil_prediksi.append(result)
    hasil_parameter.extend(detail)

# Aturan normalisasi diambil dari baseline lama; hasilnya seharusnya sama.
diff_internal = [r for r in hasil_prediksi
                 if r["full_match_internal"] !=
                 (r["function_correct"] and r["semua_nilai_normalisasi"])]
assert not diff_internal, f"Ada {len(diff_internal)} skor internal yang tidak konsisten"
assert sum(r["bfcl_ast_correct"] for r in hasil_prediksi) == 716
assert sum(r["full_match_internal"] for r in hasil_prediksi) == 701

print("Total prediksi:", len(hasil_prediksi))
print("Total baris pemeriksaan parameter:", len(hasil_parameter))
print("Distribusi kategori:")
for label, n in Counter(r["kategori_bertingkat"] for r in hasil_prediksi).most_common():
    print(f"{label}: {n}")

Total prediksi: 2937
Total baris pemeriksaan parameter: 11519
Distribusi kategori:
nilai_argumen_berbeda: 880
struktur_parameter_bermasalah: 758
cocok_teks_persis: 642
fungsi_salah: 520
kandidat_perbedaan_penulisan: 77
cocok_normalisasi_internal: 59
format_tidak_valid: 1


## 5. Ringkasan per model dan bahasa

Ringkasan memuat AST Accuracy, exact match internal, kecocokan nama fungsi, serta distribusi kesalahan argumen. Jumlah kasus dihitung per prediksi, bukan jumlah nilai parameter.

In [5]:
def ringkas(group):
    total = len(group)
    function_ok = sum(r["function_correct"] for r in group)
    strict_ok = sum(r["function_correct"] and r["semua_nilai_persis"] for r in group)
    norm_ok = sum(r["full_match_internal"] for r in group)
    out = {
        "total": total,
        "ast_benar": sum(r["bfcl_ast_correct"] for r in group),
        "ast_accuracy_pct": round(100 * sum(r["bfcl_ast_correct"] for r in group) / total, 2),
        "function_correct": function_ok,
        "function_accuracy_pct": round(100 * function_ok / total, 2),
        "teks_persis": strict_ok,
        "teks_persis_pct": round(100 * strict_ok / total, 2),
        "internal_exact": norm_ok,
        "internal_exact_pct": round(100 * norm_ok / total, 2),
        "struktur_bermasalah": sum(r["kategori_bertingkat"] == "struktur_parameter_bermasalah" for r in group),
        "nilai_berbeda": sum(r["kategori_bertingkat"] == "nilai_argumen_berbeda" for r in group),
        "kandidat_penulisan": sum(r["kategori_bertingkat"] == "kandidat_perbedaan_penulisan" for r in group),
        "ast_internal_beda": sum(r["bfcl_ast_correct"] != r["full_match_internal"] for r in group),
    }
    return out

summary = []
for model in MODELS:
    for lang in ("en-US", "id-ID", "ALL"):
        group = [r for r in hasil_prediksi if r["model"] == model and
                 (lang == "ALL" or r["language_code"] == lang)]
        summary.append({"model": model, "bahasa": lang, **ringkas(group)})

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(summary))
except ImportError:
    for s in summary:
        print(s)

,model,bahasa,total,ast_benar,ast_accuracy_pct,function_correct,function_accuracy_pct,teks_persis,teks_persis_pct,internal_exact,internal_exact_pct,struktur_bermasalah,nilai_berbeda,kandidat_penulisan,ast_internal_beda
0,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,en-US,582,155,26.63,502,86.25,145,24.91,153,26.29,160,171,18,2
1,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,id-ID,397,95,23.93,324,81.61,91,22.92,92,23.17,88,141,3,3
2,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,ALL,979,250,25.54,826,84.37,236,24.11,245,25.03,248,312,21,5
3,gemma2:9b-instruct-q4_K_M,en-US,582,134,23.02,502,86.25,115,19.76,132,22.68,194,154,22,2
4,gemma2:9b-instruct-q4_K_M,id-ID,397,70,17.63,312,78.59,63,15.87,69,17.38,122,115,6,1
5,gemma2:9b-instruct-q4_K_M,ALL,979,204,20.84,814,83.15,178,18.18,201,20.53,316,269,28,3
6,qwen2.5:7b-instruct,en-US,582,175,30.07,484,83.16,148,25.43,170,29.21,112,182,20,5
7,qwen2.5:7b-instruct,id-ID,397,87,21.91,292,73.55,80,20.15,85,21.41,82,117,8,2
8,qwen2.5:7b-instruct,ALL,979,262,26.76,776,79.26,228,23.29,255,26.05,194,299,28,7


## 6. Kandidat pemeriksaan manual

Sampel audit dipilih dari hasil salah argumen, dengan pembagian seimbang untuk model dan bahasa. Pemilihan mengutamakan variasi masalah dan `group_id` yang berbeda. Sampel ini bukan estimasi proporsi kesalahan dataset.

Kolom keputusan manual dibiarkan kosong. Perbedaan teks yang mirip, parafrase, maupun nilai `null` belum dianggap setara dengan GT tanpa penilaian manusia.

In [6]:
rng = random.Random(SEED)
sample = []
used_groups = set()
by_stratum = defaultdict(list)
for result in hasil_prediksi:
    if result["function_correct"] and not result["bfcl_ast_correct"]:
        by_stratum[(result["model"], result["language_code"])].append(result)

# Kuota per model-bahasa: format kandidat (3), struktur (3), nilai berbeda (4).
# Kuota yang tidak terpenuhi dialihkan ke kategori lain dalam strata yang sama.
quota = [
    ("kandidat_perbedaan_penulisan", 3),
    ("struktur_parameter_bermasalah", 3),
    ("nilai_argumen_berbeda", 4),
]
for key in sorted(by_stratum):
    candidates = by_stratum[key][:]
    rng.shuffle(candidates)
    chosen = []
    for label, want in quota:
        for candidate in candidates:
            if len([x for x in chosen if x["kategori_bertingkat"] == label]) >= want:
                break
            if candidate["kategori_bertingkat"] != label:
                continue
            if candidate["group_id"] in used_groups:
                continue
            chosen.append(candidate)
            used_groups.add(candidate["group_id"])
    for candidate in candidates:
        if len(chosen) >= SAMPEL_AUDIT // len(by_stratum):
            break
        if candidate["group_id"] in used_groups:
            continue
        chosen.append(candidate)
        used_groups.add(candidate["group_id"])
    if len(chosen) != SAMPEL_AUDIT // len(by_stratum):
        raise ValueError(f"Tidak cukup grup unik untuk strata {key}: {len(chosen)}")
    sample.extend(chosen)

assert len(sample) == SAMPEL_AUDIT
manual_rows = []
for i, r in enumerate(sample, 1):
    related = [x for x in hasil_parameter if x["model"] == r["model"] and x["row_key"] == r["row_key"]]
    problems = [x for x in related if x["kategori"] not in ("teks_persis", "opsional_tidak_diisi")]
    manual_rows.append({
        "no": i, "model": r["model"], "bahasa": r["language_code"],
        "row_key": r["row_key"], "group_id": r["group_id"],
        "instruksi": r["question"], "fungsi": r["target_function"],
        "ground_truth": r["ground_truth_json"], "prediksi": r["prediction_arguments_json"],
        "kategori_otomatis": r["kategori_bertingkat"],
        "parameter_perlu_ditinjau": "; ".join(f"{x['parameter']} ({x['kategori']})" for x in problems),
        "ast_error_type": r["ast_error_type"],
        "keputusan_manual": "", "alasan_manual": "", "sumber_verifikasi": "",
    })
print("Sampel:", len(manual_rows))
print("Komposisi:", dict(Counter((r["model"], r["bahasa"]) for r in manual_rows)))

Sampel: 60
Komposisi: {('Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m', 'en-US'): 10, ('Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m', 'id-ID'): 10, ('gemma2:9b-instruct-q4_K_M', 'en-US'): 10, ('gemma2:9b-instruct-q4_K_M', 'id-ID'): 10, ('qwen2.5:7b-instruct', 'en-US'): 10, ('qwen2.5:7b-instruct', 'id-ID'): 10}


## 7. Penyimpanan hasil

Hasil tersimpan di folder `results/evaluasi_argumen_bertingkat`. File audit memisahkan temuan otomatis dan keputusan manual. Seluruh data sumber tetap utuh.

In [7]:
def simpan_csv(filename, records):
    if not records:
        raise ValueError(f"Tidak ada data untuk {filename}")
    path = OUTPUT / filename
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    return path


files = [
    simpan_csv("argumen_per_prediksi.csv", hasil_prediksi),
    simpan_csv("argumen_per_parameter.csv", hasil_parameter),
    simpan_csv("ringkasan_argumen_bertingkat.csv", summary),
    simpan_csv("sampel_audit_argumen_60.csv", manual_rows),
]
meta = {
    "run": "evaluasi_argumen_bertingkat_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_ast": str(INPUT),
    "record_count": len(hasil_prediksi),
    "parameter_count": len(hasil_parameter),
    "sample_count": len(manual_rows),
    "seed": SEED,
    "source_scores_unchanged": True,
    "semantic_equivalence": "manual_review_only",
    "notes": [
        "AST resmi yang sudah dihitung dipakai apa adanya, bukan dihitung ulang.",
        "Kategori bertingkat adalah diagnostik tambahan, bukan skor BFCL resmi.",
        "Sampel 60 adalah purposive stratified dan tidak dipakai sebagai estimasi prevalensi.",
        "GT dan prediksi baseline tidak diubah.",
    ],
}
meta_path = OUTPUT / "metadata_argumen_bertingkat.json"
meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
files.append(meta_path)
for path in files:
    print("Tersimpan:", path)

Tersimpan: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat\argumen_per_prediksi.csv
Tersimpan: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat\argumen_per_parameter.csv
Tersimpan: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat\ringkasan_argumen_bertingkat.csv
Tersimpan: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat\sampel_audit_argumen_60.csv
Tersimpan: D:\KP-Agentic-AI\results\evaluasi_argumen_bertingkat\metadata_argumen_bertingkat.json
